# 04 - Convolution

A CNN replaces big “all-to-all” connections with a **small sliding window**:

- A *convolution filter* is like a tiny learned feature detector (e.g., “edge looking right”)
- It slides across the image and produces a **feature map**
- Early layers learn simple patterns (edges)
- Deeper layers combine them into higher-level patterns (corners, textures, parts)
- Eventually the network predicts a class or bounding boxes

**Object detection** models (like YOLO) use CNN backbones to extract features, then heads to:
- classify objects
- predict bounding boxes
- sometimes predict keypoints/poses

Robotics connection:
- This is how you go from pixels → “note at (x,y), size w×h” in real time.

## Imports

In [4]:
import warnings
warnings.filterwarnings('ignore') # ignoring warnings for sake of space - don't do this normally! 

import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras import models, layers

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression 
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Make sure we get outputs from a probabilitistic model each time (for reproducibility) 
np.random.seed(1515)

## Convolution Basics

### What is Convolution?

**Convolution** is a mathematical operation - it involves taking a **kernel**, or smaller matrix with certain values, and "sliding" it over the image to perform a matrix operation to transform the image. For context, convolution is one kind of operation, and another similar one is called cross correlation. 

The purpose of this is to create a feature map that captures things like edges, shapes, and colors to classify an image. 

Let's see what this does to an image: 

![](./ref_imgs/convolution_squirrel.png)

You can see how the image gives us a decent idea of edges of the image. 

There are many different types of kernels! Some focus on blurring the image, some focus on color scheme. Ultimately, you can pick kernels that fit your use case best, or you can also let the kernel be "learned", in a similar way that our parameters of our models are "learned" through training methods. 

### Convolutional Neural Networks

In a **Convolutional Neural Network** or CNN, we apply this idea of a convolution to our NN framework. 

Instead of the simple functions, we use convolution functions as a layer. Let's see how this plays out with our dataset for the last notebook, CIFAR-10.

### Improvement Attempt #3: Basic CNN

In [12]:
# Read in Data Again 

# Load the MNIST dataset and split into training and testing sets
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# skip the flattening step (keep dtypes float32 for less memory) 
X_train = (X_train.astype("float32") / 255.0)
X_test  = (X_test.astype("float32") / 255.0)

y_train = y_train.reshape(-1)
y_test  = y_test.reshape(-1)

In [13]:
# Train our CNN 

cnn1 = models.Sequential()

# First convolutional block
cnn1.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
cnn1.add(layers.MaxPooling2D((2, 2)))

# regular layers 
cnn1.add(layers.Flatten()) # get our dimensions correct for regular NN implementation 
cnn1.add(layers.Dense(64, activation='relu'))
cnn1.add(layers.Dense(10, activation='softmax')) # 10 for number of labels/clases

In [14]:
# compile our model 

cnn1.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

history = cnn1.fit(X_train, y_train, epochs=10, batch_size=32, 
                    validation_data=(X_test, y_test), verbose=0)

_, cnn1_train_acc = cnn1.evaluate(X_train, y_train, verbose=0)
_, cnn1_test_acc = cnn1.evaluate(X_test, y_test, verbose=0)

print(f"CNN 1 Regression Training Accuracy: {cnn1_train_acc:3f}")
print(f"CNN 1 Regression Test Accuracy: {cnn1_test_acc:3f}")

CNN Regression Training Accuracy: 0.737040
CNN Regression Test Accuracy: 0.635300


To refresh your memory, the previous Train/Test Accuracy for the NN was 0.67/0.51, so we see some pretty drastic improvement with a very basic CNN. 

Let's add some more complexity to see how well we do.

### Final Improvement Attempt: Revamped CNN

We add some more convolution layers to perform more transformations to extract more features below:

In [ ]:
# Model Configuration

cnn2 = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)), # extra convolution layer

    layers.Conv2D(64, (3, 3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')  # 10 output classes
])

# Performance 

cnn2.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"]
)

history = cnn2.fit(X_train, y_train, epochs=10, batch_size=32, 
                    validation_data=(X_test, y_test), verbose=0)

_, cnn2_train_acc = cnn2.evaluate(X_train, y_train, verbose=0)
_, cnn2_test_acc = cnn2.evaluate(X_test, y_test, verbose=0)

print(f"CNN 2 Regression Training Accuracy: {cnn2_train_acc:3f}")
print(f"CNN 2 Regression Test Accuracy: {cnn2_test_acc:3f}")

Keep in mind, this level of performance comes without any type of tuning! So, this framework can be *very* powerful.